In [1]:
import numpy as np
import pandas as pd
import os
import re

# ---------- INPUT / OUTPUT ----------
file_path = "Users/annasve/RNA_Seq_data_antismash.xlsx"
output_folder = "Users/annasve/RNA_Seq_data_refined_borders.xlsx"
os.makedirs(output_folder, exist_ok=True)

# Load metadata + expression table
df = pd.read_excel(file_path)


In [2]:
# -------------------------------------------------
# Parameters you can tweak
# -------------------------------------------------

# Expression column detection (your media)
media_keywords = ["DNPM", "ISP2", "MA", "SoyM", "TSB", "gluc", "gly", "malt"]

# BGC refinement core parameters
expr_min           = 1.0   # logTPM threshold to be considered "expressed"
corr_threshold     = 0.6   # high co-expression with BGC eigengene
bridge_min_corr    = 0.3   # minimum corr for a "bridge" gene
max_low_run        = 3     # max number of mid-corr genes in a row while extending
max_biosyn_gap_genes = 3   # max genomic gap between biosynthetic genes in same block

# Optional merging of low-expression BGC cores
low_expr_threshold = 2.0   # logTPM; cores below this considered "silent/low"
max_gap_genes_lowexpr = 20 # max distance between low-expr cores to allow merging


In [3]:
# =================================================
# 1. Prepare df and identify columns
# =================================================

# Work on a copy; ensure deterministic genomic order
df = df.copy().reset_index(drop=True)

# Positional index for walking along the genome
df["genomic_idx"] = np.arange(len(df), dtype=int)

# --- Clean & normalize Region_Nr (handles values like 2, '2', '2_1', '10_3') ---
if "Region_Nr" not in df.columns:
    raise ValueError("Region_Nr column not found in df")

df["Region_Nr"] = df["Region_Nr"].astype(str).str.strip()
df.loc[df["Region_Nr"].isin(["", "nan", "None"]), "Region_Nr"] = np.nan

def region_sort_key(val: str):
    """Sort Region_Nr as '2', '2_1', '10_3' → (2,0), (2,1), (10,3)."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return (np.inf, np.inf, "")
    s = str(val)
    m = re.match(r"^(\d+)(?:[_\.](\d+))?$", s)
    if m:
        base = int(m.group(1))
        sub = int(m.group(2) or 0)
        return (base, sub, s)
    return (np.inf, np.inf, s)

# Expression columns (must match your media names)
expr_cols = [c for c in df.columns if any(m in c for m in media_keywords)]
if len(expr_cols) == 0:
    raise ValueError("No expression columns found. Check media_keywords and df columns.")

# Ensure Is_Core_Gene exists and is boolean
if "Is_Core_Gene" not in df.columns:
    df["Is_Core_Gene"] = False

df["Is_Core_Gene"] = (
    df["Is_Core_Gene"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["1", "true", "t", "yes"])
)

# Biosynthetic annotation: exact Gene_Kind == 'biosynthetic'
if "Gene_Kind" not in df.columns:
    raise ValueError("Gene_Kind column not found in df")

gene_kind_clean = df["Gene_Kind"].astype(str).str.strip().str.lower()
df["is_biosynthetic"] = gene_kind_clean == "biosynthetic"

max_genomic_idx = int(df["genomic_idx"].max())

# Initialise BGC-related columns (one per gene)
df["BGC_ID"]         = np.nan
df["BGC_region"]     = np.nan   # Region_Nr of the BGC
df["BGC_block"]      = np.nan   # biosynthetic block index in that region
df["corr_to_bgc"]    = np.nan
df["expressed_any"]  = False
df["high_corr"]      = False
df["refined_member"] = False

# To avoid overlap: track genes already used as seeds
assigned_genes = set()  # set of genomic_idx


In [4]:
# =================================================
# Helpers
# =================================================

def compute_eigengene(expr_mat: np.ndarray) -> np.ndarray:
    """Mean expression profile across seed genes."""
    if expr_mat.shape[0] == 0:
        return np.full(expr_mat.shape[1], np.nan)
    if expr_mat.shape[0] == 1:
        return expr_mat[0, :]
    return expr_mat.mean(axis=0)

def row_corr_to_vector(row_vals: np.ndarray, ref: np.ndarray) -> float:
    """Safe Pearson correlation between one gene and a reference profile."""
    if not np.all(np.isfinite(row_vals)) or not np.all(np.isfinite(ref)):
        return np.nan
    if np.std(row_vals) == 0 or np.std(ref) == 0:
        return np.nan
    return float(np.corrcoef(row_vals, ref)[0, 1])

def compute_stats_for_gene(idx: int, eig: np.ndarray):
    """Compute corr, expressed_any, high_corr for a single gene (by genomic_idx)."""
    vals = df.iloc[idx][expr_cols].to_numpy(dtype=float)
    expr_any = bool((vals > expr_min).any())
    corr = row_corr_to_vector(vals, eig)
    high = bool((corr >= corr_threshold) and expr_any)
    return corr, expr_any, high


In [5]:
# =================================================
# 2. Define BGC seeds and grow clusters (within Region_Nr)
# =================================================

bgc_id_counter = 1

regions = sorted(
    [r for r in df["Region_Nr"].dropna().unique()],
    key=region_sort_key,
)

for region in regions:
    region_mask = df["Region_Nr"] == region
    region_genes = df.loc[region_mask].copy()
    region_idx = np.sort(region_genes["genomic_idx"].values)

    # All biosynthetic genes in this region
    region_bio = df.loc[region_mask & df["is_biosynthetic"]].sort_values("genomic_idx")
    if region_bio.empty:
        continue

    # ---- group biosynthetic genes into blocks based on genomic distance ----
    block_ids = []
    current_block = 0
    prev_idx = None
    for idx in region_bio["genomic_idx"]:
        if prev_idx is None or (idx - prev_idx) > max_biosyn_gap_genes:
            current_block += 1
        block_ids.append(current_block)
        prev_idx = idx

    region_bio = region_bio.assign(block_id=block_ids)

    for block_id in sorted(region_bio["block_id"].unique()):
        block_rows = region_bio[region_bio["block_id"] == block_id]
        biosyn_idx = block_rows["genomic_idx"].values
        if biosyn_idx.size == 0:
            continue

        # ---- Seed genes: core biosynthetic genes first, else all biosynthetic in block ----
        block_core = block_rows[block_rows["Is_Core_Gene"]]
        if not block_core.empty:
            seed_idx_all = block_core["genomic_idx"].values
        else:
            seed_idx_all = biosyn_idx

        # Remove any seeds already assigned to another BGC (avoid overlap)
        seed_idx = [i for i in seed_idx_all if i not in assigned_genes]
        if len(seed_idx) == 0:
            continue

        seed_idx = np.array(sorted(seed_idx), dtype=int)
        seed_start = int(seed_idx.min())
        seed_end   = int(seed_idx.max())

        # ---- Eigengene from seed genes ----
        seed_expr = df.iloc[seed_idx][expr_cols].to_numpy(dtype=float)
        eig = compute_eigengene(seed_expr)

        # ---- 2a. Add all seed genes as members ----
        for idx in seed_idx:
            corr, expr_any, high = compute_stats_for_gene(idx, eig)
            df.at[idx, "corr_to_bgc"]    = corr
            df.at[idx, "expressed_any"]  = expr_any
            df.at[idx, "high_corr"]      = high
            df.at[idx, "refined_member"] = True
            df.at[idx, "BGC_ID"]         = bgc_id_counter
            df.at[idx, "BGC_region"]     = region
            df.at[idx, "BGC_block"]      = block_id

            assigned_genes.add(idx)

        # ---- 2b-LEFT. Sweep LEFT with bridge rule ----
        left_positions = [i for i in region_idx if i < seed_start][::-1]
        low_run = 0
        pending = []

        for idx in left_positions:
            if idx in assigned_genes:
                # already belongs to another BGC → hard border, stop within region
                break

            corr, expr_any, high = compute_stats_for_gene(idx, eig)
            df.at[idx, "corr_to_bgc"]   = corr
            df.at[idx, "expressed_any"] = expr_any
            df.at[idx, "high_corr"]     = high

            if not expr_any:
                break
            if corr < bridge_min_corr:
                break

            if high:
                for j in pending + [idx]:
                    df.at[j, "refined_member"] = True
                    df.at[j, "BGC_ID"]         = bgc_id_counter
                    df.at[j, "BGC_region"]     = region
                    df.at[j, "BGC_block"]      = block_id
                    assigned_genes.add(j)
                pending = []
                low_run = 0
                continue

            pending.append(idx)
            low_run += 1
            if low_run > max_low_run:
                break

        # ---- 2c-RIGHT. Sweep RIGHT with bridge rule ----
        right_positions = [i for i in region_idx if i > seed_end]
        low_run = 0
        pending = []

        for idx in right_positions:
            if idx in assigned_genes:
                break

            corr, expr_any, high = compute_stats_for_gene(idx, eig)
            df.at[idx, "corr_to_bgc"]   = corr
            df.at[idx, "expressed_any"] = expr_any
            df.at[idx, "high_corr"]     = high

            if not expr_any:
                break
            if corr < bridge_min_corr:
                break

            if high:
                for j in pending + [idx]:
                    df.at[j, "refined_member"] = True
                    df.at[j, "BGC_ID"]         = bgc_id_counter
                    df.at[j, "BGC_region"]     = region
                    df.at[j, "BGC_block"]      = block_id
                    assigned_genes.add(j)

                pending = []
                low_run = 0
                continue

            pending.append(idx)
            low_run += 1
            if low_run > max_low_run:
                break

        bgc_id_counter += 1

# First refined table
df_with_borders = df.copy()


/var/folders/zm/bd67fqqd1cg4ycfsvq5v2x_h0000gp/T/ipykernel_27130/1151742606.py:68: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1.0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "BGC_region"]     = region


In [6]:
# =================================================
# 3. Refinement: local gap-bridging and adjacent biosynthetic rule
# =================================================

dfb = df_with_borders.copy()
dfb["__orig_order__"] = np.arange(len(dfb))

# 3A. Bridge small gaps (1–3 genes) between refined genes of same BGC/Region
dfb = dfb.sort_values(["Region_Nr", "genomic_idx"]).reset_index(drop=True)

regions = dfb["Region_Nr"].dropna().unique()
for region in regions:
    region_mask = dfb["Region_Nr"] == region
    df_reg = dfb.loc[region_mask]

    for bgc in df_reg["BGC_ID"].dropna().unique():
        idxs = dfb.index[
            region_mask & (dfb["BGC_ID"] == bgc) & (dfb["refined_member"])
        ].to_list()
        if len(idxs) < 2:
            continue

        idxs = sorted(idxs)
        for k in range(len(idxs) - 1):
            i = idxs[k]
            j = idxs[k + 1]
            gap = j - i - 1
            if gap <= 0 or gap > 3:
                continue

            gap_range = range(i + 1, j)

            # don't overwrite genes already in some BGC
            if dfb.loc[list(gap_range), "BGC_ID"].notna().any():
                continue

            dfb.loc[list(gap_range), "BGC_ID"] = bgc
            dfb.loc[list(gap_range), "refined_member"] = True

# 3B. Adjacent biosynthetic rule – same BGC & merge if different
if "Geneid" not in dfb.columns:
    dfb["Geneid"] = np.arange(len(dfb), dtype=int)

dfb = dfb.sort_values("Geneid").reset_index(drop=True)

biosyn_mask = (
    dfb["Gene_Kind"].astype(str).str.strip().str.lower() == "biosynthetic"
)

n_rows = len(dfb)
for i in range(n_rows - 1):
    if not (biosyn_mask.iat[i] and biosyn_mask.iat[i + 1]):
        continue

    b1 = dfb.at[i, "BGC_ID"]
    b2 = dfb.at[i + 1, "BGC_ID"]

    if pd.isna(b1) and pd.isna(b2):
        continue

    # case 1: one has BGC_ID, the other doesn’t → copy ID
    if pd.notna(b1) and pd.isna(b2):
        dfb.loc[i + 1, "BGC_ID"] = b1
        dfb.loc[i + 1, "refined_member"] = True
    elif pd.isna(b1) and pd.notna(b2):
        dfb.loc[i, "BGC_ID"] = b2
        dfb.loc[i, "refined_member"] = True

    # case 2: both have BGC_ID but different → merge into single cluster
    elif pd.notna(b1) and pd.notna(b2) and (b1 != b2):
        try:
            new_id = min(float(b1), float(b2))
            old_id = max(float(b1), float(b2))
        except Exception:
            new_id = b1
            old_id = b2

        same_cluster_mask = dfb["BGC_ID"] == old_id
        dfb.loc[same_cluster_mask, "BGC_ID"] = new_id

        # ensure biosynthetic genes in merged cluster are refined
        dfb.loc[
            (dfb["BGC_ID"] == new_id)
            & (dfb["Gene_Kind"].astype(str).str.strip().str.lower() == "biosynthetic"),
            "refined_member"
        ] = True


In [7]:
# =================================================
# 4. OPTIONAL: merge nearby low-expression biosynthetic clusters
# =================================================

# Work in genomic order
dfb = dfb.sort_values("genomic_idx").reset_index(drop=True)
dfb["row_max_expr"] = dfb[expr_cols].max(axis=1)

gene_kind = dfb["Gene_Kind"].astype(str).str.strip().str.lower()
is_biosyn = gene_kind.eq("biosynthetic")

# Define cores as biosynthetic + low expression
low_expr_core = is_biosyn & (dfb["row_max_expr"] <= low_expr_threshold)

# Collect seeds per BGC_ID
cluster_seeds = (
    dfb[low_expr_core & dfb["BGC_ID"].notna()]
    .groupby("BGC_ID")["genomic_idx"]
    .agg(["min", "max"])
)

if not cluster_seeds.empty:
    cluster_seeds = cluster_seeds.sort_values("min")
    cluster_ids = list(cluster_seeds.index)

    # links between clusters to merge
    links = []
    for c1, c2 in zip(cluster_ids[:-1], cluster_ids[1:]):
        r1 = cluster_seeds.loc[c1]
        r2 = cluster_seeds.loc[c2]
        gap = r2["min"] - r1["max"] - 1
        if gap <= max_gap_genes_lowexpr:
            links.append((c1, c2))

    # union-find on BGC_IDs
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx == ry:
            return
        if str(rx) <= str(ry):
            parent[ry] = rx
        else:
            parent[rx] = ry

    for a, b in links:
        union(a, b)

    if parent:
        mapping = {cid: find(cid) for cid in cluster_ids}
        dfb["BGC_ID"] = dfb["BGC_ID"].map(lambda x: mapping.get(x, x))


In [8]:
# =================================================
# 5. Create refined_BGC_NR and enforce biosynthetic cores
# =================================================

bgcnum_col = "refined_BGC_NR"

# Start refined_BGC_NR from BGC_ID
dfb[bgcnum_col] = dfb["BGC_ID"]

# Genomic order
dfb = dfb.sort_values("genomic_idx").reset_index(drop=True)

gene_kind = dfb["Gene_Kind"].astype(str).str.strip().str.lower()
is_biosyn = gene_kind.eq("biosynthetic")

clusters = [c for c in dfb[bgcnum_col].dropna().unique()]

cluster_has_core = {}
for c in clusters:
    mask = dfb[bgcnum_col] == c
    cluster_has_core[c] = bool((mask & is_biosyn).any())

# For clusters without cores, merge into nearest cluster that has a core
for c in clusters:
    if cluster_has_core.get(c, False):
        continue
    cl_mask = dfb[bgcnum_col] == c
    if not cl_mask.any():
        continue

    # use median genomic position of this tailoring-only cluster
    mid_pos = dfb.loc[cl_mask, "genomic_idx"].median()

    # Find nearest cluster with a core
    best_c = None
    best_dist = np.inf
    for c2 in clusters:
        if not cluster_has_core.get(c2, False):
            continue
        c2_mask = dfb[bgcnum_col] == c2
        mid2 = dfb.loc[c2_mask, "genomic_idx"].median()
        d = abs(mid2 - mid_pos)
        if d < best_dist:
            best_dist = d
            best_c = c2

    if best_c is not None:
        dfb.loc[cl_mask, bgcnum_col] = best_c
        dfb.loc[cl_mask, "refined_member"] = True


In [9]:
# =================================================
# 6. Restore original order & generate summary
# =================================================

# Restore original order
dfb = dfb.sort_values("__orig_order__").drop(columns=["__orig_order__"]).reset_index(drop=True)

# Gene-level output
genes_out = dfb.copy()

# Simple BGC-level summary: overlap between Region_Nr and refined_BGC_NR
overlap = (
    dfb
    .assign(n=1)
    .groupby(["Region_Nr", "refined_BGC_NR", "BGC_ID"], dropna=False)["n"]
    .sum()
    .reset_index()
    .sort_values(["Region_Nr", "refined_BGC_NR", "n"], ascending=[True, True, False])
)

# =================================================
# 7. Save outputs
# =================================================

genes_out.to_excel(
    os.path.join(output_folder, "NBC_00345_genes_with_refined_bgc_borders.xlsx"),
    index=False
)

overlap.to_excel(
    os.path.join(output_folder, "NBC_00345_bgc_refined_summary.xlsx"),
    index=False
)

print("Saved:")
print("  - genes_with_refined_bgc_borders.xlsx")
print("  - bgc_refined_summary.xlsx")


Saved:
  - genes_with_refined_bgc_borders.xlsx
  - bgc_refined_summary.xlsx
